# MRI Image Denoising Demo

This notebook demonstrates the complete workflow for MRI image denoising using deep learning.

## Contents
1. Setup and Imports
2. Generate Sample Data
3. Train the Model
4. Test and Visualize Results
5. Performance Analysis

## 1. Setup and Imports

In [ ]:
import sys
sys.path.append('../src')

import os
import numpy as np
import matplotlib.pyplot as plt
import torch
from PIL import Image

from models import get_model
from dataset import MRIDataset
from metrics import calculate_psnr, calculate_ssim, calculate_mae
from utils import visualize_results, visualize_comparison

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Generate Sample Data

Generate synthetic MRI images for demonstration

In [ ]:
# Generate sample data
!cd ../src && python generate_sample_data.py --output ../data/raw --num_images 20 --size 256

## 3. Load and Visualize Data

In [ ]:
# Create dataset
dataset = MRIDataset('../data/raw', noise_level=25, patch_size=256, augment=False)

print(f"Dataset size: {len(dataset)} images")

# Visualize some samples
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    noisy, clean = dataset[i]
    
    axes[0, i].imshow(clean.squeeze(), cmap='gray')
    axes[0, i].set_title(f'Clean Image {i+1}')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(noisy.squeeze(), cmap='gray')
    axes[1, i].set_title(f'Noisy Image {i+1}')
    axes[1, i].axis('off')

plt.tight_layout()
plt.show()

## 4. Train the Model

Train DnCNN model on the sample data

In [ ]:
# Train model (quick demo with few epochs)
!cd ../src && python train.py --model dncnn --epochs 10 --batch_size 4 --lr 0.001 --train_dir ../data/raw

## 5. Load Trained Model and Test

In [ ]:
# Load trained model
model = get_model('dncnn', num_layers=17, features=64)
model = model.to(device)

# Load weights (use best model)
checkpoint_path = '../models/dncnn_epoch_10_best.pth'
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print(f"Loaded model from epoch {checkpoint['epoch']}")
else:
    print("No trained model found. Please train the model first.")

model.eval()

## 6. Denoise Images and Visualize Results

In [ ]:
# Test on sample images
num_samples = 4

for i in range(num_samples):
    # Get sample
    noisy, clean = dataset[i]
    
    # Denoise
    with torch.no_grad():
        noisy_input = noisy.unsqueeze(0).to(device)
        denoised = model(noisy_input)
        denoised = torch.clamp(denoised, 0, 1)
    
    # Convert to numpy
    clean_np = clean.squeeze().numpy()
    noisy_np = noisy.squeeze().numpy()
    denoised_np = denoised.squeeze().cpu().numpy()
    
    # Calculate metrics
    psnr = calculate_psnr(clean_np, denoised_np)
    ssim = calculate_ssim(clean_np, denoised_np)
    mae = calculate_mae(clean_np, denoised_np)
    
    metrics = {
        'psnr': psnr,
        'ssim': ssim,
        'mae': mae
    }
    
    # Visualize
    print(f"\nSample {i+1}:")
    print(f"  PSNR: {psnr:.2f} dB")
    print(f"  SSIM: {ssim:.4f}")
    print(f"  MAE:  {mae:.6f}")
    
    visualize_results(clean_np, noisy_np, denoised_np, metrics=metrics)

## 7. Performance Analysis

In [ ]:
# Evaluate on entire dataset
all_psnr = []
all_ssim = []
all_mae = []

model.eval()
with torch.no_grad():
    for i in range(len(dataset)):
        noisy, clean = dataset[i]
        
        noisy_input = noisy.unsqueeze(0).to(device)
        denoised = model(noisy_input)
        denoised = torch.clamp(denoised, 0, 1)
        
        clean_np = clean.squeeze().numpy()
        denoised_np = denoised.squeeze().cpu().numpy()
        
        psnr = calculate_psnr(clean_np, denoised_np)
        ssim = calculate_ssim(clean_np, denoised_np)
        mae = calculate_mae(clean_np, denoised_np)
        
        all_psnr.append(psnr)
        all_ssim.append(ssim)
        all_mae.append(mae)

# Print statistics
print("\n" + "="*60)
print("OVERALL PERFORMANCE")
print("="*60)
print(f"Average PSNR: {np.mean(all_psnr):.2f} ± {np.std(all_psnr):.2f} dB")
print(f"Average SSIM: {np.mean(all_ssim):.4f} ± {np.std(all_ssim):.4f}")
print(f"Average MAE:  {np.mean(all_mae):.6f} ± {np.std(all_mae):.6f}")
print("="*60)

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(all_psnr, bins=15, color='steelblue', alpha=0.7, edgecolor='black')
axes[0].set_xlabel('PSNR (dB)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('PSNR Distribution')
axes[0].axvline(np.mean(all_psnr), color='red', linestyle='--', label=f'Mean: {np.mean(all_psnr):.2f}')
axes[0].legend()

axes[1].hist(all_ssim, bins=15, color='coral', alpha=0.7, edgecolor='black')
axes[1].set_xlabel('SSIM')
axes[1].set_ylabel('Frequency')
axes[1].set_title('SSIM Distribution')
axes[1].axvline(np.mean(all_ssim), color='red', linestyle='--', label=f'Mean: {np.mean(all_ssim):.4f}')
axes[1].legend()

axes[2].hist(all_mae, bins=15, color='lightgreen', alpha=0.7, edgecolor='black')
axes[2].set_xlabel('MAE')
axes[2].set_ylabel('Frequency')
axes[2].set_title('MAE Distribution')
axes[2].axvline(np.mean(all_mae), color='red', linestyle='--', label=f'Mean: {np.mean(all_mae):.6f}')
axes[2].legend()

plt.tight_layout()
plt.show()

## 8. Conclusion

This notebook demonstrated:
- Loading and preprocessing MRI images
- Training a DnCNN model for denoising
- Evaluating performance using PSNR, SSIM, and MAE
- Visualizing results

For your thesis, you can:
1. Use real MRI datasets (BrainWeb, OASIS, etc.)
2. Train for more epochs with larger datasets
3. Compare different architectures (DnCNN vs U-Net)
4. Experiment with different noise levels
5. Analyze results for different anatomical regions